In [0]:
df = spark.read.format('csv')\
          .option("header",True)\
          .option('inferSchema',True)\
          .load("/Volumes/pyspark_dbt/source/source_data/customers")

In [0]:
display(df)

customer_id,first_name,last_name,email,phone_number,city,signup_date,last_updated_timestamp
1,Daniel,Reed,azimmerman@ramirez-nelson.com,811-724-1080,Tiffanyview,2023-12-25,2025-09-15T15:21:05.000Z
2,Jonathan,Hansen,williammiller@serrano-jones.com,632-190-4027x0198,Baldwinburgh,2021-01-23,2025-09-19T20:19:41.000Z
3,Samuel,Rodriguez,troy07@carrillo-webb.info,676.094.0716,New Ashley,2025-05-31,2025-09-13T03:45:53.000Z
4,Victor,Sanchez,jonthompson@thomas.biz,960-929-2694,Stephanieton,2024-08-31,2025-08-23T00:38:03.000Z
5,Sabrina,Black,patricknewman@williams.biz,+1-621-699-9458x79470,South Christopherport,2023-05-18,2025-08-20T16:08:38.000Z
6,Melissa,Blair,courtney29@gmail.com,(332)469-2812x91807,East Pamela,2025-09-17,2025-08-24T09:44:20.000Z
7,Jacqueline,Williams,robertmoore@gmail.com,7312603248,Fritzmouth,2023-05-29,2025-08-28T19:40:31.000Z
8,Christina,Arnold,parkerbridget@yahoo.com,(878)682-1357,North Sarah,2023-09-21,2025-09-06T20:36:02.000Z
9,Lisa,Shelton,cody49@johnson.org,9349468560,South Kyleview,2022-12-13,2025-09-08T14:57:58.000Z
10,Megan,Dean,vritter@yahoo.com,001-967-061-8100x482,Port Jesse,2023-03-12,2025-09-07T23:05:32.000Z


In [0]:
customer_schema = df.schema   
customer_schema

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:725)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:443)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:443)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

### SPARK STREAMING

In [0]:
entities = ['customers','drivers','locations','payments','trips','vehicles']

In [0]:
for entity in entities:

    df_batch = spark.read.format('csv')\
          .option("header",True)\
          .option('inferSchema',True)\
          .load(f"/Volumes/pyspark_dbt/source/source_data/{entity}")

    schema_entity = df_batch.schema

    df = spark.readStream.format('csv')\
                     .option('header',True)\
                     .schema(schema_entity)\
                     .load(f"/Volumes/pyspark_dbt/source/source_data/{entity}")

    df.writeStream.format('delta')\
          .outputMode('append')\
          .option("checkpointLocation", f"/Volumes/pyspark_dbt/bronze/checkpoint/{entity}")\
          .trigger(once=True)\
          .toTable(f"pyspark_dbt.bronze.{entity}")